In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from econml.dml import CausalForestDML
import joblib
import os, sys, warnings
from const_paths import SAVE_DIR

warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.expanduser('~/my_ukb_thesis'))

phase_II_dir = os.path.expanduser('~/my_ukb_thesis/phase_II')
os.makedirs(phase_II_dir, exist_ok=True)
os.makedirs(os.path.join(phase_II_dir, 'outputs'), exist_ok=True)
os.makedirs(os.path.join(phase_II_dir, 'figures'), exist_ok=True)

print(f"Phase II directory: {phase_II_dir}")
print(f"Outputs: {os.path.join(phase_II_dir, 'outputs')}")
print(f"Figures: {os.path.join(phase_II_dir, 'figures')}")

PHASE2_DIR  = os.path.expanduser('~/my_ukb_thesis/phase_II')
OUTPUT_DIR  = os.path.join(PHASE2_DIR, 'outputs')
FIGURES_DIR = os.path.join(PHASE2_DIR, 'figures')

print("Libraries loaded.")
print(f"SAVE_DIR: {SAVE_DIR}")

/home/rmhihbp/.conda/envs/ukb_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'const_paths'

In [ ]:
data_dir   = SAVE_DIR
target_col = 'def_CVD_AF_HF_AFTER'

# Load the training data for split B
df_B = pd.read_parquet(os.path.join(data_dir, 'split_B_train.parquet'))

print(f"Split B shape: {df_B.shape}")
print(f"\nColumns:")
print(list(df_B.columns))

Split B shape: (298245, 28)

Columns:
['eid', 'def_CVD_AF_HF_AFTER', 'diet_total', 'sleep_hrs', 'PA_active', 'device_pc', 'mental_doctor', 'mental_risk', 'age_defined_baseline', 'genetic_sex', 'BMI', 'FH_cvd_f', 'FH_cvd_m', 'FH_cvd_sib', 'uni_degree', 'MobUse_≤1yr', 'MobUse_2-4yr', 'MobUse_5-8yr', 'MobUse_>8yr', 'smk_prev', 'smk_curr', 'alc_prev', 'alc_curr', 'ProcMeat_<1wk', 'ProcMeat_1wk', 'ProcMeat_2-4wk', 'ProcMeat_5-6wk', 'ProcMeat_daily']


In [ ]:
CONFOUNDERS = [
    'age_defined_baseline',
    'genetic_sex',
    'BMI',
    'uni_degree',
    'FH_cvd_f',
    'FH_cvd_m',
    'FH_cvd_sib',
    'mental_doctor',
    'alc_curr',
]

TREATMENTS = [
    'smk_curr',
    'PA_active',
    'sleep_adequate',  
]

OUTCOME = 'def_CVD_AF_HF_AFTER'

# check if all required columns are present in the dataframe (except for sleep_adequate)
cols_to_check = CONFOUNDERS + ['smk_curr', 'PA_active', OUTCOME]
missing = [c for c in cols_to_check if c not in df_B.columns]

if missing:
    print(f"WARNING - Missing columns: {missing}")
else:
    print("All required columns found.")

print(f"\nConfounders ({len(CONFOUNDERS)}): {CONFOUNDERS}")
print(f"Treatments  ({len(TREATMENTS)}): {TREATMENTS}")
print(f"Outcome: {OUTCOME}")

All required columns found.

Confounders (9): ['age_defined_baseline', 'genetic_sex', 'BMI', 'uni_degree', 'FH_cvd_f', 'FH_cvd_m', 'FH_cvd_sib', 'mental_doctor', 'alc_curr']
Treatments  (3): ['smk_curr', 'PA_active', 'sleep_adequate']
Outcome: def_CVD_AF_HF_AFTER


In [ ]:
# use float for sleep_adequate to allow NaN

df_B['sleep_adequate'] = (df_B['sleep_hrs'] >= 7).astype(float)
df_B.loc[df_B['sleep_hrs'].isna(), 'sleep_adequate'] = np.nan

# check the distribution of sleep_hrs and sleep_adequate, and count NaN values
print(f"sleep_hrs NaN: {df_B['sleep_hrs'].isna().sum()}")
print(f"sleep_adequate NaN: {df_B['sleep_adequate'].isna().sum()}")

# print the distribution of sleep_adequate, excluding NaN values
print(f"\nsleep_adequate distribution (excluding NaN):")
counts = df_B['sleep_adequate'].value_counts().sort_index()
for val, count in counts.items():
    pct = count / df_B['sleep_adequate'].notna().sum() * 100
    label = 'Adequate (>=7h)  ' if val == 1.0 else 'Inadequate (<7h) '
    print(f"  {label}: {count:,} ({pct:.1f}%)")

sleep_hrs NaN: 1813
sleep_adequate NaN: 1813

sleep_adequate distribution (excluding NaN):
  Inadequate (<7h) : 72,408 (24.4%)
  Adequate (>=7h)  : 224,024 (75.6%)


In [ ]:
print("=" * 50)
print("NaN Diagnostic")
print("=" * 50)

# For each confounder and treatment, count the number of NaN values and their percentage.
print("\nW (Confounders):")
for col in CONFOUNDERS:
    n = df_B[col].isna().sum()
    flag = " ← HAS NaN" if n > 0 else ""
    print(f"  {col:25s}: {n:,} ({n/len(df_B)*100:.2f}%){flag}")

# For each treatment, count the number of NaN values and their percentage.
print("\nT (Treatments):")
for col in TREATMENTS:
    n = df_B[col].isna().sum()
    flag = " ← HAS NaN" if n > 0 else ""
    print(f"  {col:25s}: {n:,} ({n/len(df_B)*100:.2f}%){flag}")

# For the outcome, count the number of NaN values and their percentage.
print(f"\nY (Outcome):")
n = df_B[OUTCOME].isna().sum()
print(f"  {OUTCOME:25s}: {n:,}")

NaN Diagnostic

W (Confounders):
  age_defined_baseline     : 0 (0.00%)
  genetic_sex              : 0 (0.00%)
  BMI                      : 0 (0.00%)
  uni_degree               : 0 (0.00%)
  FH_cvd_f                 : 0 (0.00%)
  FH_cvd_m                 : 0 (0.00%)
  FH_cvd_sib               : 0 (0.00%)
  mental_doctor            : 0 (0.00%)
  alc_curr                 : 913 (0.31%) ← HAS NaN

T (Treatments):
  smk_curr                 : 0 (0.00%)
  PA_active                : 0 (0.00%)
  sleep_adequate           : 1,813 (0.61%) ← HAS NaN

Y (Outcome):
  def_CVD_AF_HF_AFTER      : 0


In [ ]:
# ── Step 1: impute W（alc_curr 有 NaN，需要处理）─────────
W_df_raw = df_B[CONFOUNDERS].copy()

w_nan_cols = [c for c in CONFOUNDERS if W_df_raw[c].isna().sum() > 0]
print(f"W columns with NaN: {w_nan_cols}")

if w_nan_cols:
    w_imputer = IterativeImputer(max_iter=10, random_state=42)
    W_df_imp = pd.DataFrame(
        w_imputer.fit_transform(W_df_raw),
        columns=CONFOUNDERS,
        index=W_df_raw.index
    )
    # alc_curr 是二元变量，round + clip
    W_df_imp['alc_curr'] = np.clip(np.round(W_df_imp['alc_curr']), 0, 1)
    joblib.dump(w_imputer, os.path.join(OUTPUT_DIR, 'confounder_imputer.pkl'))
    print("W imputation done.")
else:
    W_df_imp = W_df_raw.copy()
    print("No NaN in W, skipping imputation.")

print(f"W NaN after: {W_df_imp.isna().sum().sum()}")

# ── Step 2: impute T（只有 sleep_adequate 有 NaN）────────
T_df_raw = df_B[TREATMENTS].copy()

t_imputer = IterativeImputer(max_iter=10, random_state=42)
T_imp = pd.DataFrame(
    t_imputer.fit_transform(T_df_raw),
    columns=TREATMENTS,
    index=T_df_raw.index
)
for col in TREATMENTS:
    T_imp[col] = np.clip(np.round(T_imp[col]), 0, 1)

joblib.dump(t_imputer, os.path.join(OUTPUT_DIR, 'treatment_imputer.pkl'))

print(f"\nT NaN after: {T_imp.isna().sum().sum()}")
print(f"T value check:")
for col in TREATMENTS:
    print(f"  {col:25s}: {sorted(T_imp[col].unique())}")

W columns with NaN: ['alc_curr']


W imputation done.
W NaN after: 0

T NaN after: 0
T value check:
  smk_curr                 : [np.float64(0.0), np.float64(1.0)]
  PA_active                : [np.float64(0.0), np.float64(1.0)]
  sleep_adequate           : [np.float64(0.0), np.float64(1.0)]


In [ ]:
# standardize age_defined_baseline and BMI, and save the scaler

print("Before imputation of age/BMI:")
print(W_df_imp[['age_defined_baseline', 'BMI']].describe().round(3).loc[['mean','std']])

scaler = StandardScaler()
W_df_imp[['age_defined_baseline', 'BMI']] = scaler.fit_transform(
    W_df_imp[['age_defined_baseline', 'BMI']])

joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'confounder_scaler.pkl'))

# check the distribution of age and BMI after standardization
print("W after standardization:")
print(W_df_imp[['age_defined_baseline', 'BMI']].describe().round(3))

# check the unique values of binary confounders to ensure they are still binary after imputation (except for age and BMI which are continuous)
print(f"\nBinary confounders (no standardization):")
binary_confounders = [c for c in CONFOUNDERS 
                    if c not in ['age_defined_baseline', 'BMI']]
for col in binary_confounders:
    print(f"  {col:25s}: {sorted(W_df_imp[col].unique())}")

Before imputation of age/BMI:
      age_defined_baseline     BMI
mean                56.207  27.193
std                  8.106   4.647
W after standardization:
       age_defined_baseline         BMI
count            298245.000  298245.000
mean                 -0.000       0.000
std                   1.000       1.000
min                  -2.370      -3.131
25%                  -0.766      -0.685
50%                   0.098      -0.140
75%                   0.838       0.518
max                   1.948       8.987

Binary confounders (no standardization):
  genetic_sex              : [np.float64(0.0), np.float64(1.0)]
  uni_degree               : [np.float64(0.0), np.float64(1.0)]
  FH_cvd_f                 : [np.float64(0.0), np.float64(1.0)]
  FH_cvd_m                 : [np.float64(0.0), np.float64(1.0)]
  FH_cvd_sib               : [np.float64(0.0), np.float64(1.0)]
  mental_doctor            : [np.float64(0.0), np.float64(1.0)]
  alc_curr                 : [np.float64(0.0), np.floa

In [ ]:
# generate final W, T, Y arrays for modeling
W = W_df_imp.values
T = T_imp.values
Y = df_B[OUTCOME].values

print(f"W shape: {W.shape}  ← (n_samples, {len(CONFOUNDERS)} confounders)")
print(f"T shape: {T.shape}  ← (n_samples, {len(TREATMENTS)} treatments)")
print(f"Y shape: {Y.shape}  ← (n_samples,)")

print(f"\nFinal NaN check:")
print(f"  W: {np.isnan(W).sum()}")
print(f"  T: {np.isnan(T).sum()}")
print(f"  Y: {np.isnan(Y).sum()}")

print(f"\nColumn order:")
print(f"  W: {CONFOUNDERS}")
print(f"  T: {TREATMENTS}")
print(f"  Y: {OUTCOME}")

W shape: (298245, 9)  ← (n_samples, 9 confounders)
T shape: (298245, 3)  ← (n_samples, 3 treatments)
Y shape: (298245,)  ← (n_samples,)

Final NaN check:
  W: 0
  T: 0
  Y: 0

Column order:
  W: ['age_defined_baseline', 'genetic_sex', 'BMI', 'uni_degree', 'FH_cvd_f', 'FH_cvd_m', 'FH_cvd_sib', 'mental_doctor', 'alc_curr']
  T: ['smk_curr', 'PA_active', 'sleep_adequate']
  Y: def_CVD_AF_HF_AFTER


In [ ]:
# sanity check: run CausalForestDML on a small subset to ensure it works without errors and produces reasonable ITE estimates
N_TEST = 5000

W_small = W[:N_TEST]
Y_small = Y[:N_TEST]

print(f"Running sanity check on {N_TEST} samples...")
print(f"Testing {len(TREATMENTS)} treatments independently...\n")

ite_test = {}

for col in TREATMENTS:
    T_single = T_imp[col].values[:N_TEST]
    
    model = CausalForestDML(
        model_y=GradientBoostingRegressor(
            n_estimators=50, max_depth=3, random_state=42
        ),
        model_t=GradientBoostingRegressor(
            n_estimators=50, max_depth=3, random_state=42
        ),
        n_estimators=100,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1,
        verbose=0,
    )
    
    model.fit(Y_small, T_single, X=W_small)
    ite_test[col] = model.effect(W_small)
    
    print(f"  {col:25s}: shape={ite_test[col].shape}, "
        f"mean={ite_test[col].mean():.4f}, "
        f"std={ite_test[col].std():.4f}")

print(f"\nSanity check PASSED!")
print(f"\nITE direction interpretation:")
print(f"  smk_curr   : ITE > 0 → smoking increases risk → intervention (1→0) decreases risk")
print(f"  PA_active  : ITE < 0 → being active decreases risk → intervention (0→1) decreases risk")
print(f"  sleep_adequate: ITE < 0 → adequate sleep decreases risk → intervention (0→1) decreases risk")

Running sanity check on 5000 samples...
Testing 3 treatments independently...

  smk_curr                 : shape=(5000,), mean=0.0423, std=0.0431
  PA_active                : shape=(5000,), mean=-0.0105, std=0.0251
  sleep_adequate           : shape=(5000,), mean=-0.0217, std=0.0315

Sanity check PASSED!

ITE direction interpretation:
  smk_curr   : ITE > 0 → smoking increases risk → intervention (1→0) decreases risk
  PA_active  : ITE < 0 → being active decreases risk → intervention (0→1) decreases risk
  sleep_adequate: ITE < 0 → adequate sleep decreases risk → intervention (0→1) decreases risk


In [ ]:
# save the full df_B with imputed W and T for later use in heterogeneity analysis
df_B.to_parquet(os.path.join(OUTPUT_DIR, 'split_B_phase2.parquet'), index=False)
joblib.dump(w_imputer, os.path.join(OUTPUT_DIR, 'confounder_imputer.pkl'))
joblib.dump(t_imputer, os.path.join(OUTPUT_DIR, 'treatment_imputer.pkl'))
joblib.dump(scaler,    os.path.join(OUTPUT_DIR, 'confounder_scaler.pkl'))

print("Saved:")
print(f"  confounder_imputer.pkl ← W imputation (alc_curr)")
print(f"  treatment_imputer.pkl  ← T imputation (sleep_adequate)")
print(f"  confounder_scaler.pkl  ← W standardization")
print(f"  split_B_phase2.parquet ← full df for heterogeneity analysis")
print("\nNotebook 1 complete. Ready for Notebook 2.")

Saved:
  confounder_imputer.pkl ← W imputation (alc_curr)
  treatment_imputer.pkl  ← T imputation (sleep_adequate)
  confounder_scaler.pkl  ← W standardization
  split_B_phase2.parquet ← full df for heterogeneity analysis

Notebook 1 complete. Ready for Notebook 2.
